# Your first Earth Engine script (Python)

**Fullstack Remote Sensing** by Rifky Nauval Hendrawan · [Chapter: Navigating the Ecosystem](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/en/05-ecosystem.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/id/05-ecosystem.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch05_first_script)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Fullstack-Remote-Sensing.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch05_first_script.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

CHAPTER 5 | Load a global elevation model, ask what it is, and draw it.

In Colab:
    import ee
    ee.Authenticate()
    ee.Initialize(project="your-cloud-project")

In [ ]:
import ee

# 1. Load a single image from the public catalogue
dem = ee.Image("CGIAR/SRTM90_V4")

# 2. Ask the server what it is holding. In Python you ask explicitly with
#    getInfo(); in the Code Editor, print() does the same round trip.
#    print(dem.getInfo())

# 3-4. Two renderings of the same numbers
grey = {"bands": ["elevation"], "min": 0, "max": 1200}
palette = {"bands": ["elevation"], "min": 0, "max": 1200,
           "palette": ["blue", "cyan", "green", "yellow", "red", "brown"]}

# 5. Longitude first. The same two numbers in the wrong order land somewhere else.
right = ee.Geometry.Point([117.161, -0.53])    # (longitude, latitude): Kalimantan

In [ ]:
# Swapped, [-0.53, 117.161] would ask for latitude 117°, which does not exist.


def elevation_at(point, label):
    value = dem.reduceRegion(ee.Reducer.first(), point, 90).get("elevation")
    return ee.Feature(None, {"point": label, "elevation_m": value})


borneo = ee.Geometry.Rectangle([108.5, -4.5, 119.5, 7.5])

In [ ]:
def products():
    return [
        {"kind": "map", "name": "ch05-dem-grey", "image": dem, "vis": grey,
         "region": borneo, "title": "SRTM elevation, greyscale",
         "source": "SRTM 90 m v4 (CGIAR-CSI). Google Earth Engine.",
         "caption": "The same data as the next map, stretched from 0 m (black) to "
                    "1,200 m (white). It answers 'how high', but slowly."},
        {"kind": "map", "name": "ch05-dem-palette", "image": dem, "vis": palette,
         "region": borneo, "legend": "Elevation (m)", "title": "SRTM elevation, palette",
         "source": "SRTM 90 m v4 (CGIAR-CSI). Google Earth Engine.",
         "caption": "A palette over the same 0 to 1,200 m range. The central highlands "
                    "of Borneo separate from the coastal lowlands at a glance."},
        {"kind": "table", "name": "ch05-lonlat", "data": ee.FeatureCollection([
            elevation_at(right, "[117.161, -0.53]  longitude first"),
            elevation_at(ee.Geometry.Point([117.161, 0.53]), "[117.161, 0.53]  sign of latitude flipped")]),
         "columns": ["point", "elevation_m"],
         "caption": "One flipped sign moves the point about 120 km. Swapping the two "
                    "numbers entirely gives a latitude of 117°, which does not exist."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())